# 05. Análisis usando inteligencia artificial

### Dataset bancario limpio

**Objetivo de este notebook:** tomar las métricas y agrupaciones ya calculadas en Python sobre `dataset_bancario_limpio.xlsx` y usarlas como insumo para que un modelo de IA (Gemini) genere un **reporte ejecutivo** y una **clasificación de alertas priorizadas**.

Regla de oro del flujo: **Python calcula, Gemini interpreta.** Nunca le enviamos las 150 filas crudas al modelo — solo resúmenes y agrupaciones ya validados, para que no pueda inventar cifras.

Cada bloque de este notebook sigue la misma lógica que el notebook 03:

1. Una celda de **instrucciones** (qué vamos a hacer y por qué).
2. El **código** que genera el cálculo, el resumen o la llamada a la IA.
3. Una celda de **insight / conclusión parcial** con la lectura de esos resultados.

Al final (sección 9) consolidamos todo en **conclusiones ejecutivas y próximos pasos**.

## 1. Instalar librerías necesarias

Instalamos `google-genai` (SDK oficial de Gemini) y `python-dotenv` (para leer la API key desde un archivo `.env` sin exponerla en el notebook). Ejecuta esta celda una sola vez.

In [1]:
# !pip install google-genai python-dotenv pandas openpyxl

## 2. Cargar el dataset limpio

Partimos del mismo archivo que usamos en el notebook 03 (`dataset_bancario_limpio.xlsx`). No lo modificamos: aquí solo lo leemos para recalcular los resúmenes que le vamos a enviar a la IA.

In [1]:
import os
import json
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from google import genai

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

RUTA_DATASET = "dataset_bancario_limpio.xlsx"

df_limpio = pd.read_excel(RUTA_DATASET)

print("Dimensiones del dataset:", df_limpio.shape)
df_limpio.head()

Dimensiones del dataset: (150, 33)


,transaccion_id,fecha_operacion,fecha_contable,empresa_id,empresa,cuenta_bancaria,banco,tipo_movimiento,categoria,proveedor_cliente,documento_ref,moneda,valor_bruto,impuesto_iva,valor_neto,metodo_pago,centro_costo,ciudad,pais,responsable,estado_conciliacion,fecha_vencimiento,dias_mora,nivel_riesgo,canal,observaciones,requiere_revision,motivo_revision,regla_signo_valida,regla_valor_neto_valida,regla_fechas_valida,regla_mora_valida,bandera_duplicado_conciliacion
0,TRX-001,2026-01-07,2026-01-07,EMP-002,Logistica Prisma Ltda.,1002-553311,Banco Capital,Egreso,Nomina,Servicios Beta,NOM-20260001,COP,-156933,0.00,-156933.00,Transferencia local,Comercial,Medellin,Colombia,Carlos Pena,Conciliado,2025-12-31,0,Bajo,PSE,NaN,False,No aplica,True,True,True,True,False
1,TRX-002,2026-01-10,2026-01-10,EMP-003,Insumos Horizonte S.A.,1003-902244,Banco Pacifico,Egreso,Impuestos,Aduanas Global,PAG-20260002,COP,-228866,0.00,-228866.00,Tarjeta credito,Operaciones,Cali,Colombia,Diana Torres,Conciliado,2026-01-08,0,Bajo,Transferencia,NaN,False,No aplica,True,True,True,True,False
2,TRX-003,2026-01-13,2026-01-13,EMP-001,Nova Retail S.A.S.,1001-884422,Banco Nacional,Egreso,Servicios publicos,Energia Urbana,PAG-20260003,COP,-300799,-57151.81,-357950.81,Cheque,Tecnologia,Barranquilla,Colombia,Felipe Mora,Conciliado,2026-01-16,0,Bajo,Tarjeta,NaN,False,No aplica,True,True,True,True,False
3,TRX-004,2026-01-16,2026-01-17,EMP-002,Logistica Prisma Ltda.,1002-553311,Banco Andino,Egreso,Arriendo,Talento Humano,PAG-20260004,COP,-372732,-70819.08,-443551.08,Debito automatico,Finanzas,Cartagena,Colombia,Sin asignar,Pendiente,2026-01-24,159,Medio,Debito automatico,NaN,True,Conciliacion no cerrada,True,True,True,True,False
4,TRX-005,2026-01-19,2026-01-19,EMP-003,Insumos Horizonte S.A.,1003-902244,Banco Capital,Ingreso,Credito bancario,Arrendadora Centro,FAC-20260005,COP,444665,0.00,444665.00,ACH,Sin asignar,Bogota,Colombia,Ana Ruiz,Conciliado,2026-02-01,0,Bajo,Caja,NaN,False,No aplica,True,True,True,True,False


**Insight:** el dataset conserva las 150 transacciones y las 33 columnas ya validadas en el notebook 03 (reglas de signo, de valor neto, de fechas y de mora). Confirmamos que la carga es correcta antes de resumir nada.

## 3. Resumen general (KPIs)

Instrucción: construimos un diccionario con los indicadores globales del periodo — volumen de transacciones, valor neto total, mezcla ingreso/egreso, mora y estado de conciliación. Este es el primer bloque que le enviaremos a Gemini.

In [2]:
resumen_general = {
    "total_transacciones": len(df_limpio),
    "valor_total_neto": float(df_limpio["valor_neto"].sum()),
    "ingresos": int((df_limpio["tipo_movimiento"] == "Ingreso").sum()),
    "egresos": int((df_limpio["tipo_movimiento"] == "Egreso").sum()),
    "mora_promedio": float(df_limpio["dias_mora"].mean()),
    "casos_mora_critica": int((df_limpio["dias_mora"] > 30).sum()),
    "pendientes": int((df_limpio["estado_conciliacion"] == "Pendiente").sum()),
    "rechazados": int((df_limpio["estado_conciliacion"] == "Rechazado").sum()),
    "diferencias": int((df_limpio["estado_conciliacion"] == "Diferencia").sum()),
}

resumen_general

{'total_transacciones': 150,
 'valor_total_neto': -165197731.94,
 'ingresos': 57,
 'egresos': 93,
 'mora_promedio': 32.03333333333333,
 'casos_mora_critica': 45,
 'pendientes': 33,
 'rechazados': 7,
 'diferencias': 11}

**Insight:** de 150 transacciones, 93 son egresos y 57 ingresos, con un valor neto total negativo (más salidas que entradas de caja en el periodo). La mora promedio general es de ~32 días, pero hay 45 casos con mora crítica (>30 días) — el 30% del dataset —, y 33 transacciones siguen pendientes de conciliar. Esto ya anticipa que el problema no está distribuido parejo, lo confirmamos en las siguientes agrupaciones.

## 4. Agrupación por empresa

Instrucción: agrupamos por empresa para ver si el riesgo de mora y el volumen de valor neto están concentrados en alguna de las tres compañías del dataset.

In [4]:
por_empresa = df_limpio.groupby("empresa").agg(
    transacciones=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
    mora_promedio=("dias_mora", "mean"),
).reset_index().sort_values("valor_total")

por_empresa

,empresa,transacciones,valor_total,mora_promedio
1,Logistica Prisma Ltda.,50,-59583015.10,29.10
0,Insumos Horizonte S.A.,50,-55831110.42,38.38
2,Nova Retail S.A.S.,50,-49783606.42,28.62


**Insight:** las tres empresas tienen un volumen parejo (50 transacciones cada una), pero **Insumos Horizonte S.A.** tiene la mora promedio más alta (38.4 días) y también el valor neto más negativo (-$55.8M). No es una diferencia dramática entre empresas — el problema de mora no parece ser por cliente/empresa, lo que sugiere mirar el corte por banco.

## 5. Agrupación por banco

Instrucción: repetimos el mismo cálculo agrupando por banco, para aislar si el problema de mora está asociado a una entidad bancaria en particular (por ejemplo, demoras en su proceso de conciliación).

In [5]:
por_banco = df_limpio.groupby("banco").agg(
    transacciones=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
    mora_promedio=("dias_mora", "mean"),
).reset_index().sort_values("mora_promedio", ascending=False)

por_banco

,banco,transacciones,valor_total,mora_promedio
0,Banco Andino,37,-49478840.24,91.027027
3,Banco Pacifico,38,-30176371.92,13.447368
1,Banco Capital,38,-44848882.92,13.131579
2,Banco Nacional,37,-40693636.86,11.540541


**Insight (hallazgo clave):** **Banco Andino** tiene una mora promedio de **91 días**, muy por encima de los demás bancos (11 a 13 días). No es un problema generalizado del dataset — está concentrado casi por completo en una sola entidad. Esto es justo el tipo de hallazgo que vale la pena resumir para la IA, en vez de mandarle las 150 filas.

## 6. Agrupación por estado de conciliación

Instrucción: medimos cuántas transacciones y cuánto valor hay en cada estado (Conciliado, Pendiente, Diferencia, Rechazado, Duplicado), y su mora promedio asociada.

In [6]:
por_estado = df_limpio.groupby("estado_conciliacion").agg(
    transacciones=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
    mora_promedio=("dias_mora", "mean"),
).reset_index()

por_estado

,estado_conciliacion,transacciones,valor_total,mora_promedio
0,Conciliado,95,-91385485.57,0.000000
1,Diferencia,11,-11819394.12,79.181818
2,Duplicado,4,-17808115.96,77.000000
3,Pendiente,33,-27732137.64,95.151515
4,Rechazado,7,-16452598.65,69.428571


**Insight:** el 63% de las transacciones (95 de 150) ya están conciliadas, con mora promedio de 0 días, como era de esperarse. El resto — Pendiente (33), Diferencia (11), Rechazado (7) y Duplicado (4) — concentra toda la mora crítica, con promedios entre 69 y 95 días. Es decir: el problema de mora *es* el problema de conciliación pendiente, no un fenómeno aparte.

## 7. Agrupación por categoría

Instrucción: vemos qué categorías de movimiento (ventas, nómina, arriendo, impuestos, etc.) concentran el mayor valor, para entender la composición del flujo de caja.

In [7]:
por_categoria = df_limpio.groupby("categoria").agg(
    transacciones=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
).reset_index().sort_values("valor_total", ascending=False)

por_categoria

,categoria,transacciones,valor_total
9,Venta nacional,20,97492242.26
8,Venta internacional,19,85391458.00
2,Credito bancario,28,22809877.00
3,Impuestos,12,-48398360.00
1,Comision bancaria,12,-51275680.00
7,Servicios publicos,12,-51773230.04
6,Reembolso,12,-52138876.00
4,Nomina,12,-52331452.00
5,Pago proveedor,12,-53485235.44
0,Arriendo,11,-61488475.72


**Insight:** 

los ingresos vienen casi todos de Venta nacional (+97.5M) y Venta internacional (+85.4M), junto con Crédito bancario (+$22.8M). 

Los egresos están repartidos de forma más pareja entre Arriendo, Pago proveedor, Nómina, Servicios públicos, Comisión bancaria e Impuestos, todos entre -48M y -61M. No hay una sola categoría de gasto que explique el déficit neto; es estructural.

## 8. Tablas cruzadas: banco × estado y empresa × nivel de riesgo

Instrucción: usamos `crosstab` para confirmar visualmente el hallazgo de la sección 5 (¿Banco Andino concentra los pendientes?) y para ver cómo se distribuye el nivel de riesgo entre empresas.

In [8]:
cruce_banco_estado = pd.crosstab(df_limpio["banco"], df_limpio["estado_conciliacion"])
cruce_banco_estado

estado_conciliacion,Conciliado,Diferencia,Duplicado,Pendiente,Rechazado
banco,,,,,
Banco Andino,0,2,1,33,1
Banco Capital,32,3,1,0,2
Banco Nacional,31,3,1,0,2
Banco Pacifico,32,3,1,0,2


In [9]:
cruce_empresa_riesgo = pd.crosstab(df_limpio["empresa"], df_limpio["nivel_riesgo"])
cruce_empresa_riesgo

nivel_riesgo,Alto,Bajo,Medio
empresa,,,
Insumos Horizonte S.A.,2,32,16
Logistica Prisma Ltda.,2,34,14
Nova Retail S.A.S.,3,35,12


**Insight:** el cruce lo confirma sin ambigüedad — **las 33 transacciones "Pendiente" del dataset pertenecen todas a Banco Andino**, y esa misma entidad concentra también la mayoría de sus Diferencias y Rechazos. Los otros tres bancos tienen un patrón casi idéntico entre sí (mayoría Conciliado, 2-3 Diferencias, 1-2 Rechazados). En cuanto al riesgo por empresa, el nivel "Alto" es minoritario y parejo (2-3 casos por empresa); el riesgo "Medio" es algo mayor en Insumos Horizonte (16 casos), consistente con su mora promedio más alta.

## 9. Consolidar el resumen estructurado

Instrucción: reunimos todo lo anterior en un solo diccionario (`analisis`) en formato JSON. Este es el resumen que efectivamente le vamos a mandar a Gemini — no el dataset completo.

In [10]:
analisis = {
    "resumen_general": resumen_general,
    "por_empresa": por_empresa.to_dict(orient="records"),
    "por_banco": por_banco.to_dict(orient="records"),
    "por_estado_conciliacion": por_estado.to_dict(orient="records"),
    "por_categoria": por_categoria.to_dict(orient="records"),
}

print(json.dumps(analisis, ensure_ascii=False, indent=2)[:1200], "...")

{
  "resumen_general": {
    "total_transacciones": 150,
    "valor_total_neto": -165197731.94,
    "ingresos": 57,
    "egresos": 93,
    "mora_promedio": 32.03333333333333,
    "casos_mora_critica": 45,
    "pendientes": 33,
    "rechazados": 7,
    "diferencias": 11
  },
  "por_empresa": [
    {
      "empresa": "Logistica Prisma Ltda.",
      "transacciones": 50,
      "valor_total": -59583015.1,
      "mora_promedio": 29.1
    },
    {
      "empresa": "Insumos Horizonte S.A.",
      "transacciones": 50,
      "valor_total": -55831110.42,
      "mora_promedio": 38.38
    },
    {
      "empresa": "Nova Retail S.A.S.",
      "transacciones": 50,
      "valor_total": -49783606.42,
      "mora_promedio": 28.62
    }
  ],
  "por_banco": [
    {
      "banco": "Banco Andino",
      "transacciones": 37,
      "valor_total": -49478840.24,
      "mora_promedio": 91.02702702702703
    },
    {
      "banco": "Banco Pacifico",
      "transacciones": 38,
      "valor_total": -30176371.92,
  

**Insight:** el JSON final es compacto (unas pocas decenas de líneas) frente a las 150 filas x 33 columnas originales. Esto reduce costo y tiempo de respuesta del modelo, y — más importante — evita que la IA tenga que "adivinar" patrones que ya calculamos con certeza en Python.

## 10. Configurar la API key de Gemini

Instrucción: cargamos la API key desde un archivo `.env` (no se escribe directamente en el notebook). 

In [13]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("No se encontro GEMINI_API_KEY. Revisa tu archivo .env")

client = genai.Client(api_key=api_key)
print("Cliente de Gemini inicializado correctamente.")

Cliente de Gemini inicializado correctamente.


## 11. Generar el reporte ejecutivo con IA

Instrucción: le pedimos a Gemini que, usando **únicamente** el JSON de la sección 9, redacte un reporte ejecutivo con hallazgos, riesgos, alertas, recomendaciones y conclusión. Le indicamos explícitamente que no invente datos.

In [14]:
prompt = f"""
Actua como analista financiero y de conciliacion bancaria.

Tengo un dataset bancario empresarial ya limpio y analizado en Python.
Con base en las siguientes metricas y agrupaciones, genera:

1. Resumen ejecutivo
2. Hallazgos principales
3. Riesgos operativos
4. Alertas prioritarias
5. Recomendaciones concretas
6. Conclusion final

No inventes datos. Usa unicamente la informacion entregada.

Datos:
{json.dumps(analisis, ensure_ascii=False, indent=2)}
"""

respuesta = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt,
)

print(respuesta.text)

Estimado equipo directivo,

A continuación, presento el informe de análisis financiero y de conciliación bancaria elaborado a partir del dataset empresarial procesado. 

---

### 1. Resumen Ejecutivo

El dataset analiza un total de **150 transacciones** distribuidas equitativamente entre tres filiales (50 transacciones cada una), con un **valor total neto de -165,197,731.94**. Esta cifra negativa se explica por un mayor volumen de egresos (93 transacciones) frente a ingresos (57 transacciones). 

A nivel operativo, la salud de la conciliación muestra que, si bien la mayoría de las transacciones están conciliadas (95 casos), existe un volumen crítico de transacciones con problemas de gestión: **33 pendientes, 11 diferencias, 7 rechazadas y 4 duplicadas**. La mora promedio general se sitúa en **32.03**, impulsada principalmente por retrasos severos en Banco Andino y en los estados "Pendiente", "Diferencia" y "Duplicado". Se identifican **45 casos de mora crítica** que requieren intervenc

## 12. Guardar el reporte ejecutivo

Instrucción: guardamos la respuesta del modelo como archivo `.md`, con fecha en el nombre, para tener un historial de reportes generados.

In [15]:
fecha_hoy = datetime.now().strftime("%Y-%m-%d_%H%M")
nombre_reporte = f"reporte_ejecutivo_{fecha_hoy}.md"

with open(nombre_reporte, "w", encoding="utf-8") as f:
    f.write(respuesta.text)

print(f"Reporte guardado en: {nombre_reporte}")

Reporte guardado en: reporte_ejecutivo_2026-07-03_2139.md


**Insight:** cada corrida queda con su propio archivo con timestamp, así que si automatizamos esto con n8n (sección 15) podemos acumular un histórico de reportes sin sobrescribir el anterior.

## 13. Filtrar casos de alerta

Instrucción: aislamos las transacciones problemáticas (mora > 30 días, o estado Pendiente/Diferencia/Rechazado) para clasificarlas con IA. Limitamos a 30 casos para no sobrecargar el prompt.

In [19]:
casos_alerta = df_limpio[
    (df_limpio["dias_mora"] > 30)
    | (df_limpio["estado_conciliacion"].isin(["Pendiente", "Diferencia", "Rechazado"]))
].copy()

casos_alerta_resumen = casos_alerta[
    [
        "transaccion_id",
        "empresa",
        "banco",
        "estado_conciliacion",
        "valor_neto",
        "dias_mora",
        "nivel_riesgo",
        "motivo_revision",
    ]
].head(30).to_dict(orient="records")

print(f"Casos de alerta encontrados: {len(casos_alerta)}")
casos_alerta_resumen[:3]

Casos de alerta encontrados: 54


[{'transaccion_id': 'TRX-004',
  'empresa': 'Logistica Prisma Ltda.',
  'banco': 'Banco Andino',
  'estado_conciliacion': 'Pendiente',
  'valor_neto': -443551.08,
  'dias_mora': 159,
  'nivel_riesgo': 'Medio',
  'motivo_revision': 'Conciliacion no cerrada'},
 {'transaccion_id': 'TRX-008',
  'empresa': 'Nova Retail S.A.S.',
  'banco': 'Banco Andino',
  'estado_conciliacion': 'Pendiente',
  'valor_neto': -785952.16,
  'dias_mora': 127,
  'nivel_riesgo': 'Medio',
  'motivo_revision': 'Conciliacion no cerrada'},
 {'transaccion_id': 'TRX-012',
  'empresa': 'Logistica Prisma Ltda.',
  'banco': 'Banco Andino',
  'estado_conciliacion': 'Pendiente',
  'valor_neto': -1128353.24,
  'dias_mora': 140,
  'nivel_riesgo': 'Medio',
  'motivo_revision': 'Conciliacion no cerrada'}]

## 14. Clasificar alertas por prioridad con IA

Instrucción: le pedimos a Gemini que tome cada caso de alerta y lo clasifique en prioridad Alta, Media o Baja, con motivo y acción recomendada — esto es más rápido de auditar manualmente que 45 filas sueltas en Excel.

In [20]:
prompt_alertas = f"""
Actúa como analista de riesgo bancario. Clasifica estos casos en prioridad Alta, Media o Baja.

Devuelve tu respuesta ESTRICTAMENTE en formato JSON válido (una lista de diccionarios). 
No incluyas texto antes ni después.
Usa exactamente estas claves: "transaccion_id", "prioridad", "motivo", "accion_recomendada".

Casos:
{json.dumps(casos_alerta_resumen, ensure_ascii=False, indent=2)}
"""

respuesta_alertas = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt_alertas,
)

print(respuesta_alertas.text)

[
  {
    "transaccion_id": "TRX-004",
    "prioridad": "Media",
    "motivo": "Mora prolongada de 159 días bajo estado Pendiente, aunque el impacto financiero es moderado.",
    "accion_recomendada": "Contactar al gestor de cuentas de Banco Andino para acelerar el cierre de la conciliación."
  },
  {
    "transaccion_id": "TRX-008",
    "prioridad": "Media",
    "motivo": "Mora de 127 días en estado Pendiente con un saldo pasivo moderado-alto.",
    "accion_recomendada": "Requerir a Nova Retail S.A.S. el soporte de la transacción para su cotejo inmediato."
  },
  {
    "transaccion_id": "TRX-012",
    "prioridad": "Alta",
    "motivo": "Mora crítica de 140 días y saldo deudor que supera el millón de unidades monetarias.",
    "accion_recomendada": "Iniciar proceso de aclaración formal con Banco Andino y exigir la regularización del saldo."
  },
  {
    "transaccion_id": "TRX-013",
    "prioridad": "Alta",
    "motivo": "Estado de 'Diferencia' sin resolver por más de 130 días con saldo

In [21]:
# Guardar el reporte de la IA en un archivo
nombre_alertas = f"clasificacion_alertas_{fecha_hoy}.md"

with open(nombre_alertas, "w", encoding="utf-8") as f:
    f.write(respuesta_alertas.text)

print(f"Clasificacion de alertas guardada en: {nombre_alertas}")

Clasificacion de alertas guardada en: clasificacion_alertas_2026-07-03_2139.md


## 15. Conclusiones ejecutivas y próximos pasos

**Consolidado de todo el análisis:**

- El dataset tiene un valor neto total negativo en el periodo — más egresos que ingresos —, sin que ninguna categoría de gasto en particular explique el déficit.
- El problema de mora **no** está repartido parejo entre empresas ni es estructural del negocio: está **concentrado casi por completo en Banco Andino**, que acumula el 100% de los pendientes y una mora promedio de 91 días frente a 11-13 días en los demás bancos.
- 54 de 150 transacciones requieren seguimiento activo por mora crítica o estado de conciliación problemático.
- El reporte ejecutivo y la clasificación de alertas generados por Gemini quedaron guardados como archivos `.md`, lo que permite mantener un historial auditable de cada corrida.

**Próximo paso:** conectar este notebook a **n8n** para que se ejecute automáticamente (por ejemplo, cada vez que llegue un nuevo `dataset_bancario_limpio.xlsx`, o en un horario fijo diario/semanal), y que el reporte y la clasificación de alertas se envíen automáticamente por correo o al equipo responsable.